# 05 过拟合与正则化

## 目标
看见「训练变好、验证变差」能说出过拟合；知道 Dropout 与 weight_decay 的使用阶段。

## 会错的直觉
「再多训几个 epoch 总会更好。」——过了某点只是在背训练集。

In [ ]:
from mnist_lab.tiny_data import make_tiny_loaders
from mnist_lab.models import build_model
from mnist_lab.train import train_model
tr, va, te = make_tiny_loaders(batch_size=32)
plain = train_model(build_model('cnn'), tr, va, epochs=4, lr=0.01, run_dir='runs/nb05_plain')
reg = train_model(build_model('cnn_dropout'), tr, va, epochs=4, lr=0.01, weight_decay=1e-3, run_dir='runs/nb05_reg')
print('plain val', plain.history['val_acc'])
print('dropout+wd val', reg.history['val_acc'])
assert 'val_loss' in plain.history

In [ ]:
from exercises.solutions.ex04_overfit import is_overfitting
assert is_overfitting([1.0, 0.2], [1.0, 1.6]) is True
assert is_overfitting([1.0, 0.7], [1.0, 0.8]) is False
print('过拟合检测器工作正常')

看训练/验证损失是否同向。

In [ ]:
print('plain train', plain.history['train_loss'])
print('plain val_loss', plain.history['val_loss'])
assert len(plain.history['train_loss']) == len(plain.history['val_loss'])

Dropout 只在 train() 时随机丢。

In [ ]:
m = build_model('cnn_dropout')
m.train()
a = m(torch.ones(1, 1, 28, 28)).detach()
b = m(torch.ones(1, 1, 28, 28)).detach()
m.eval()
c = m(torch.ones(1, 1, 28, 28)).detach()
d = m(torch.ones(1, 1, 28, 28)).detach()
print('eval 两次是否相同', torch.allclose(c, d))
assert torch.allclose(c, d)

## 思考题
1. Dropout 在 `model.eval()` 时还随机丢神经元吗？
2. 早停看的是哪条曲线？
3. 数据增强算正则吗？它和 Dropout 惩罚的是同一件事吗？